# RAG Example

In this example we build and export a Chroma Database that can be used on a local LLM execution for RAG.
The database is created against a Github repo passed as parameter.

In [7]:
import shutil
import hashlib
import sentence_transformers
from langchain_community.document_loaders import DirectoryLoader,TextLoader
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from git import Repo
from langchain_chroma import Chroma

In [ ]:
files_pattern = "**/*.py,**/*.ts,**/*.tsx,**/*.md"
repo = "kubeflow/kale"
git_url = "https://github.com"
embeddings_model = "all-MiniLM-L6-v2"

In [ ]:
# Clone the repo and save the zipped content
repo_dir = '/tmp/repo'
repo_url = f'{git_url}/{repo}'
shutil.rmtree(repo_dir, ignore_errors=True)
Repo.clone_from(repo_url, repo_dir)
loader = DirectoryLoader(repo_dir, glob=files_pattern.split(","),loader_cls=TextLoader,use_multithreading=True)
docs = loader.load()
print(f'Finished reading {len(docs)} docs from repo {repo_url}')

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name=embeddings_model)

In [ ]:
db_dir = "/tmp/chroma_db"
Chroma.from_documents(
    documents=docs, 
    embedding=embeddings, 
    persist_directory=db_dir
)
zip_file = shutil.make_archive('/tmp/chroma_db', 'zip', db_dir)
with open(zip_file, 'rb') as f:
    chroma_db = f.read()

In [ ]:
md5_hash = hashlib.md5(chroma_db).hexdigest()
print(f'Vector DB created sucessfully. MD5 Hash: {md5_hash}')